# Notebook 5 — Regresor LSTM (Pronóstico de Precios)
**Proyecto:** Ernesto Investing AI — iDeSo

Entrena un regresor LSTM univariado (precio de cierre) para los 5 tickers del proyecto y guarda los resultados en MongoDB (colección `predicciones_lstm`): históricos de validación (real vs. predicho) y una proyección autoregresiva a 30 días con bandas de confianza al 95%.

Igual que el Notebook 4, lee de `precios_ohlcv` (poblada por el Notebook 1) en vez de volver a descargar de yfinance.

## 1. Instalación e importación de librerías

In [ ]:
!pip install -q tensorflow scikit-learn pymongo[srv] certifi

import numpy as np
import pandas as pd
import json
import os
import math
from datetime import datetime, timezone
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout

np.random.seed(42)
tf.random.set_seed(42)
print("Entorno de regresión configurado.")

## 2. Conexión a MongoDB Atlas

In [ ]:
from google.colab import userdata
import certifi
from pymongo import MongoClient

MONGO_URI = userdata.get('MONGO_URI')

cliente = MongoClient(MONGO_URI, tlsCAFile=certifi.where())
db = cliente['ernesto_investing_ai']

col_precios = db['precios_ohlcv']        # poblada por el Notebook 1 (entrada)
col_lstm = db['predicciones_lstm']       # la puebla este notebook (salida)
col_lstm.create_index('ticker', unique=True)

cliente.admin.command('ping')
print('✅ Conexión a MongoDB Atlas exitosa')

## 3. Configuración común

In [ ]:
TICKERS = ["FSM", "VOLCABC1.LM", "ABX.TO", "BVN", "BHP"]
VENTANA = 60          # 60 días de historia para predecir el día siguiente
PROYECCION_DIAS = 30  # horizonte de pronóstico a futuro
EPOCHS = 60
PORCENTAJE_TRAIN = 0.8


def cargar_precios(ticker: str) -> pd.DataFrame:
    cursor = col_precios.find({'ticker': ticker}).sort('fecha', 1)
    return pd.DataFrame(list(cursor))

## 4. Entrenamiento, validación y proyección futura por ticker

In [ ]:
def entrenar_lstm_para_ticker(ticker: str) -> dict:
    df = cargar_precios(ticker)
    if df.empty or len(df) < (VENTANA + 60):
        print(f"  ⚠️ {ticker}: datos insuficientes en MongoDB, se omite.")
        return None

    precios_reales = df['Close'].dropna().values.reshape(-1, 1)
    scaler = MinMaxScaler(feature_range=(0, 1))
    precios_scaled = scaler.fit_transform(precios_reales)

    X, y = [], []
    for i in range(len(precios_scaled) - VENTANA):
        X.append(precios_scaled[i:i + VENTANA])
        y.append(precios_scaled[i + VENTANA])
    X, y = np.array(X), np.array(y)

    split = int(len(X) * PORCENTAJE_TRAIN)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    modelo = Sequential([
        LSTM(64, return_sequences=True, input_shape=(VENTANA, 1)),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1, activation='linear'),
    ])
    modelo.compile(optimizer='adam', loss='mean_squared_error')
    modelo.fit(X_train, y_train, epochs=EPOCHS, batch_size=32, verbose=0)

    pred_scaled = modelo.predict(X_test, verbose=0)
    pred_usd = scaler.inverse_transform(pred_scaled).flatten()
    y_test_usd = scaler.inverse_transform(y_test).flatten()

    rmse = float(np.sqrt(mean_squared_error(y_test_usd, pred_usd)))
    mae = float(mean_absolute_error(y_test_usd, pred_usd))
    r2 = float(r2_score(y_test_usd, pred_usd))
    rmse_pct = float((rmse / np.mean(y_test_usd)) * 100)

    fechas_reales = pd.to_datetime(df['fecha']).iloc[-len(y_test_usd):].dt.strftime('%Y-%m-%d').tolist()
    historico_validacion = [
        {"fecha": f, "real": round(float(r), 4), "predicho": round(float(p), 4)}
        for f, r, p in zip(fechas_reales, y_test_usd, pred_usd)
    ]

    # Proyección autoregresiva a futuro: cada predicción se re-alimenta como entrada
    ultima_ventana = precios_scaled[-VENTANA:].tolist()
    predicciones_futuras_scaled = []
    for _ in range(PROYECCION_DIAS):
        entrada = np.array([ultima_ventana[-VENTANA:]])
        pred_nueva = modelo.predict(entrada, verbose=0)[0][0]
        predicciones_futuras_scaled.append(pred_nueva)
        ultima_ventana.append([pred_nueva])

    precios_futuros = scaler.inverse_transform(
        np.array(predicciones_futuras_scaled).reshape(-1, 1)
    ).flatten()

    ultima_fecha = pd.to_datetime(df['fecha']).iloc[-1]
    proyeccion_futura = []
    for paso, precio in enumerate(precios_futuros, start=1):
        ancho_banda = 1.96 * rmse * math.sqrt(paso)
        fecha_pred = ultima_fecha + pd.Timedelta(days=paso)
        while fecha_pred.weekday() >= 5:  # saltar fines de semana
            fecha_pred += pd.Timedelta(days=1)
        proyeccion_futura.append({
            "fecha": fecha_pred.strftime('%Y-%m-%d'),
            "prediccion_usd": round(float(precio), 4),
            "banda_min": round(float(precio - ancho_banda), 4),
            "banda_max": round(float(precio + ancho_banda), 4),
        })

    print(f"  {ticker}: RMSE={rmse:.3f} USD ({rmse_pct:.2f}%) | R²={r2:.3f}")

    return {
        "ticker": ticker,
        "metricas_error": {
            "rmse_usd": round(rmse, 4),
            "rmse_porcentaje": round(rmse_pct, 2),
            "mae_usd": round(mae, 4),
            "r2_score": round(r2, 4),
        },
        "historico_validacion": historico_validacion,
        "proyeccion_futura": proyeccion_futura,
    }

## 5. Ejecutar para los 5 tickers y guardar en MongoDB (+ respaldo JSON local)

In [ ]:
os.makedirs('data', exist_ok=True)
resultados_todos = {}

for ticker in TICKERS:
    print(f"\n=== Entrenando regresor LSTM para {ticker} ===")
    contrato = entrenar_lstm_para_ticker(ticker)
    if contrato is None:
        continue

    col_lstm.update_one(
        {'ticker': ticker},
        {'$set': {**contrato, 'created_at': datetime.now(timezone.utc)}},
        upsert=True,
    )
    resultados_todos[ticker] = contrato

with open('data/predicciones_lstm.json', 'w') as f:
    json.dump(resultados_todos, f, indent=2, ensure_ascii=False)

print(f"\n✅ Listo. {len(resultados_todos)}/{len(TICKERS)} tickers guardados en 'predicciones_lstm' "
      f"y en data/predicciones_lstm.json")
print("Nota: el horizonte de proyección quedó fijo en 30 días al guardarse pre-calculado en Mongo "
      "(a diferencia de la versión backend/, que puede recalcular cualquier horizonte al vuelo).")